## 1. Before You Begin

Use this notebook to test locale-specific voices, documented code-switching pairs, and chunked narration with MAI-Voice-2. MAI-Voice is in public preview and is not recommended for production workloads.

| Detail | Value |
|---|---|
| Released | 2026-06-02 |
| Model card | [MAI-Voice-2](https://ai.azure.com/catalog/models/MAI-Voice-2) |
| Pricing | Billed per character; check [current Speech pricing](https://azure.microsoft.com/pricing/details/speech/) |
| Setup | Complete [models/quickstart/](../../quickstart/README.md), then create a supported Speech resource |

Generated PCM WAV files are written under `output/`. The comparisons are listening exercises, not objective quality benchmarks.

In [ ]:
## 2. Verify your environment
%pip install azure-cognitiveservices-speech python-dotenv --quiet

import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

load_dotenv()
required = ["MICROSOFT_FOUNDRY_ENDPOINT", "MICROSOFT_FOUNDRY_API_KEY", "AZURE_SPEECH_ENDPOINT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Complete models/quickstart/ and add AZURE_SPEECH_ENDPOINT before continuing."
    )

parsed = urlparse(os.environ["AZURE_SPEECH_ENDPOINT"])
if parsed.scheme != "https" or not parsed.netloc:
    raise ValueError("AZURE_SPEECH_ENDPOINT must be an HTTPS resource endpoint.")

SPEECH_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
SPEECH_KEY = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Environment ready; audio will be written to {OUTPUT_DIR.resolve()}")

## 3. Compare locale-specific voices

Synthesize semantically matched welcome lines with voices designed for each locale. Do not compare accent alone: listen for intelligibility, rhythm, emphasis, and whether punctuation produces equivalent pauses. Confirm voice availability in the live table before adapting the list.

In [ ]:
## 4. Build a multilingual listening gallery
import html
import time
import wave

import azure.cognitiveservices.speech as speechsdk
import pandas as pd
from IPython.display import Audio, display

def build_ssml(text, *, voice, locale, style=None, style_degree=None):
    safe_text = html.escape(text)
    if style is None:
        body = safe_text
    else:
        degree = "" if style_degree is None else f' styledegree="{style_degree}"'
        body = f'<mstts:express-as style="{style}"{degree}>{safe_text}</mstts:express-as>'
    return (
        f'<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" '
        f'xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="{locale}">'
        f'<voice name="{voice}">{body}</voice></speak>'
    )

def wav_duration(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getnframes() / audio.getframerate()

def synthesize(text, filename, *, voice, locale, style=None, style_degree=None):
    path = OUTPUT_DIR / filename
    config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, endpoint=SPEECH_ENDPOINT)
    config.speech_synthesis_voice_name = voice
    config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm)
    output = speechsdk.audio.AudioOutputConfig(filename=str(path))
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=config, audio_config=output)
    started = time.perf_counter()
    if style is None:
        result = synthesizer.speak_text_async(text).get()
    else:
        ssml = build_ssml(text, voice=voice, locale=locale, style=style, style_degree=style_degree)
        result = synthesizer.speak_ssml_async(ssml).get()
    elapsed = time.perf_counter() - started
    if result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
        details = result.cancellation_details
        raise RuntimeError(f"Synthesis canceled: {details.reason}; {details.error_details}")
    return {"path": path, "seconds": wav_duration(path), "request_seconds": elapsed}

samples = [
    ("en-US", "en-US-Ethan:MAI-Voice-2", "Welcome. Your museum tour begins in the central gallery."),
    ("es-MX", "es-MX-Valeria:MAI-Voice-2", "Bienvenidos. Su recorrido por el museo comienza en la galería central."),
    ("fr-FR", "fr-FR-Soleil:MAI-Voice-2", "Bienvenue. Votre visite du musée commence dans la galerie centrale."),
    ("de-DE", "de-DE-Mia:MAI-Voice-2", "Willkommen. Ihr Museumsrundgang beginnt in der zentralen Galerie."),
    ("hi-IN", "hi-IN-Kavya:MAI-Voice-2", "स्वागत है। आपका संग्रहालय भ्रमण केंद्रीय दीर्घा से शुरू होता है।"),
]

gallery = []
for index, (locale, voice, text) in enumerate(samples, start=1):
    result = synthesize(text, f"01-gallery-{index:02d}-{locale}.wav", voice=voice, locale=locale)
    gallery.append({"locale": locale, "voice": voice, "text": text, **result})

display(pd.DataFrame(gallery)[["locale", "voice", "seconds", "request_seconds"]])
for row in gallery:
    print(f'{row["locale"]}: {row["text"]}')
    display(Audio(filename=str(row["path"])))

## 5. Test code-switching

The MAI-Voice-2 announcement specifically calls out Hindi-English and Spanish-English code-switching. These samples keep each language in a natural conversational role. Listen at the switch boundary for dropped words, an abrupt persona change, or unnatural timing.

In [ ]:
## 6. Render controlled code-switched prompts
code_switch_samples = [
    (
        "hi-en",
        "hi-IN-Dhruv:MAI-Voice-2",
        "hi-IN",
        "आज की workshop में हम responsible AI के तीन practical checks सीखेंगे।",
    ),
    (
        "es-en",
        "es-MX-Valeria:MAI-Voice-2",
        "es-MX",
        "La reunión empieza a las nueve, so please bring the final design notes.",
    ),
]

switch_rows = []
for label, voice, locale, text in code_switch_samples:
    result = synthesize(text, f"02-code-switch-{label}.wav", voice=voice, locale=locale)
    switch_rows.append({"pair": label, "voice": voice, "text": text, **result})
    print(f"{label}: {text}")
    display(Audio(filename=str(result["path"])))

display(pd.DataFrame(switch_rows)[["pair", "voice", "seconds", "request_seconds"]])

## 7. Assemble a long-form narration

Long text should be chunked at semantic boundaries instead of arbitrary character counts. The next exercise keeps one voice and WAV format across four paragraphs, records each request, and joins the segments. Review pacing and persona consistency at every boundary.

In [ ]:
## 8. Synthesize and join narration chunks
narration = [
    "At dawn, the field station woke to the sound of rain moving across the roof.",
    "The overnight sensors had captured an unusual temperature shift along the wetlands.",
    "Before collecting new samples, the team compared the readings with the previous week and marked two locations for inspection.",
    "By noon, the weather had cleared, and the revised map was ready for the next survey crew.",
]
voice = "en-US-Ethan:MAI-Voice-2"
locale = "en-US"
chunk_rows = []
for index, paragraph in enumerate(narration, start=1):
    result = synthesize(paragraph, f"03-narration-{index:02d}.wav", voice=voice, locale=locale)
    chunk_rows.append({"chunk": index, "text": paragraph, **result})

def join_wavs(paths, destination):
    frames = []
    expected = None
    for path in paths:
        with wave.open(str(path), "rb") as source:
            params = source.getparams()
            signature = (params.nchannels, params.sampwidth, params.framerate, params.comptype)
            if expected is None:
                expected = signature
            elif signature != expected:
                raise ValueError(f"Incompatible WAV parameters in {path}")
            frames.append(source.readframes(source.getnframes()))
    with wave.open(str(destination), "wb") as target:
        target.setnchannels(expected[0])
        target.setsampwidth(expected[1])
        target.setframerate(expected[2])
        target.setcomptype(expected[3], "not compressed")
        for chunk in frames:
            target.writeframes(chunk)
    return destination

full_narration = join_wavs(
    [row["path"] for row in chunk_rows],
    OUTPUT_DIR / "03-complete-narration.wav",
)
display(pd.DataFrame(chunk_rows)[["chunk", "seconds", "request_seconds", "text"]])
print(f"Joined narration: {wav_duration(full_narration):.2f} seconds")
display(Audio(filename=str(full_narration)))

## 9. Your Turn to Explore

1. Add another supported locale using a native translation and a locale-matched prebuilt voice.
2. Rewrite a code-switched prompt so the language transition occurs at a different phrase boundary.
3. Split the narration into sentences instead of paragraphs and compare boundary continuity and total request time.

## 10. Summary

You compared locale-specific prebuilt voices, tested two documented code-switching pairs, and assembled a paragraph-chunked narration for continuity review. Use MAI-Voice-2 for multilingual or extended content where fidelity matters; use MAI-Voice-2-Flash when low latency is the primary requirement. See the [Audio / Speech primer](../../../docs/primers/audio-speech.md) and the repository [glossary](../../../docs/GLOSSARY.md) for related terminology.

## 11. References

- [MAI-Voice overview](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-voices) — prebuilt voice table, multilingual support, SSML controls, and preview status.
- [MAI-Voice-2 model card](https://ai.azure.com/catalog/models/MAI-Voice-2) — long-form, multilingual, and voice-prompting capability notes.
- [Introducing MAI-Voice-2](https://microsoft.ai/news/mai-voice-2/) — supported language list and documented Hindi-English and Spanish-English code-switching.
- [Speech synthesis with the Speech SDK](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/get-started-text-to-speech) — SDK setup and audio output.
- [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions) — endpoint and regional availability guidance.